# 01 — Data audit

Coverage and quality checks on the downloaded NBA seasons: market counts, candle/trade coverage, spread environment, tip-time inference validation.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import pandas as pd
import matplotlib.pyplot as plt

SPORT, SEASON = 'nba', '2024-25'
base = pathlib.Path('..') / 'data' / 'raw' / SPORT / SEASON
markets = pd.read_parquet(base / 'markets.parquet')
manifest = pd.read_parquet(base / 'manifest.parquet')
print(len(markets), 'markets;', manifest['status'].value_counts().to_dict())
markets[['ticker','open_time','close_time','result','volume']].head()

In [ ]:
# Volume distribution across markets (log scale): where the liquidity lives
ax = markets['volume'].clip(lower=1).plot.hist(bins=80, logx=False, logy=True, figsize=(8,4))
ax.set_xlabel('lifetime contracts traded'); ax.set_title('Market volume distribution');

In [ ]:
# Pregame spread environment + tip inference on a sample of games
from kalshi_mm.data.build import iter_games
rows = []
for i, g in enumerate(iter_games('../data/raw', SPORT, SEASON, pregame_hours=6)):
    s = g.stream.dropna(subset=['bid','ask'])
    if s.empty: continue
    rows.append({'ticker': g.ticker, 'tip_source': g.tip_source,
                 'median_spread': (s['ask']-s['bid']).median(),
                 'n_trades': int((g.stream['etype']=='trade').sum()),
                 'tip_hour_utc': g.tip_ts.hour + g.tip_ts.minute/60})
    if i >= 400: break
audit = pd.DataFrame(rows)
print(audit['tip_source'].value_counts(normalize=True).round(3).to_dict())
print('median pregame spread (c):', audit['median_spread'].median())
audit['tip_hour_utc'].plot.hist(bins=48, figsize=(8,3), title='Inferred tip hour (UTC) — NBA games cluster 23:00–03:30Z as expected');

**Checks:** tip hours should cluster at NBA start times (7–10:30pm ET ≈ 23:00–03:30 UTC); `detected` share should dominate `fallback`; median pregame spread ≈ 1¢.